# Cyber Attack Detection — Exploratory Data Analysis

**Notebook:** `01_eda.ipynb`
**Dataset:** `data/processed/clean.parquet` (CIC-IDS2017, preprocessed)
**Purpose:** Understand class balance, identify the most informative features, visualise
how benign and attack traffic differ, and expose correlations between features.

---

### Sections
1. Setup and Data Load
2. Class Distribution
3. Top 15 Features by Correlation with `is_attack`
4. Feature Distributions: Normal vs Attack
5. Correlation Heatmap of Top Features

## 1. Setup and Data Load

In [ ]:
"""
Cell 1: Imports and global plot settings.
All paths are relative to the project root so this notebook
works on any machine without editing absolute paths.
"""

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns

warnings.filterwarnings('ignore')

# Global colour palette
PALETTE_BINARY  = ['#2ECC71', '#E74C3C']   # green=benign, red=attack
PALETTE_CLASSES = ['#2ECC71', '#E74C3C', '#3498DB', '#F39C12',
                   '#9B59B6', '#1ABC9C', '#E67E22']

# Dark theme
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi'      : 120,
    'figure.facecolor': '#0F1117',
    'axes.facecolor'  : '#1A1D27',
    'axes.edgecolor'  : '#3A3D4D',
    'axes.labelcolor' : '#E0E0E0',
    'xtick.color'     : '#C0C0C0',
    'ytick.color'     : '#C0C0C0',
    'text.color'      : '#E0E0E0',
    'grid.color'      : '#2A2D3D',
    'grid.linestyle'  : '--',
    'grid.alpha'      : 0.5,
})

# Path setup: notebook lives in notebooks/, project root is one level up
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PARQUET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'clean.parquet'

print(f'Project root : {PROJECT_ROOT}')
print(f'Data file    : {PARQUET_PATH}')
print(f'File exists  : {PARQUET_PATH.exists()}')

In [ ]:
"""
Cell 2: Load the cleaned Parquet file and do a quick sanity check.
"""

df = pd.read_parquet(PARQUET_PATH)

LABEL_COLS   = ['Label', 'is_attack', 'attack_type']
FEATURE_COLS = [c for c in df.columns if c not in LABEL_COLS]
NUM_COLS     = df[FEATURE_COLS].select_dtypes(include='number').columns.tolist()

print(f'Dataset shape    : {df.shape[0]:,} rows  x  {df.shape[1]} columns')
print(f'Feature columns  : {FEATURE_COLS}')
print(f'Numeric features : {NUM_COLS}')
print()
df.head()

---
## 2. Class Distribution

In [ ]:
"""
Cell 3: Two side-by-side bar charts.
  Left  : Binary split (Benign vs Attack)
  Right : Per-attack-type counts
"""

binary_counts = df['is_attack'].value_counts().sort_index()
binary_labels = ['Benign', 'Attack']
multi_counts  = df['Label'].value_counts().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Class Distribution', fontsize=15, fontweight='bold', y=1.02)

# Left: Binary
ax0 = axes[0]
bars0 = ax0.bar(binary_labels, binary_counts.values,
                color=PALETTE_BINARY, edgecolor='white',
                linewidth=0.8, width=0.45)
total = binary_counts.sum()
for bar, count in zip(bars0, binary_counts.values):
    ax0.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.01,
        f'{count:,}\n({100*count/total:.1f}%)',
        ha='center', va='bottom', fontsize=10, color='white', fontweight='bold'
    )
ax0.set_title('Binary: Benign vs Attack', fontsize=12, pad=10)
ax0.set_ylabel('Number of Samples')
ax0.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax0.set_ylim(0, binary_counts.max() * 1.25)

# Right: Multi-class
ax1 = axes[1]
colors1 = PALETTE_CLASSES[:len(multi_counts)]
bars1 = ax1.bar(multi_counts.index, multi_counts.values,
                color=colors1, edgecolor='white', linewidth=0.8)
for bar, count in zip(bars1, multi_counts.values):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.005,
        f'{count:,}\n({100*count/total:.1f}%)',
        ha='center', va='bottom', fontsize=9, color='white', fontweight='bold'
    )
ax1.set_title('Per Attack-Type Breakdown', fontsize=12, pad=10)
ax1.set_ylabel('Number of Samples')
ax1.set_xlabel('Traffic Class')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.set_ylim(0, multi_counts.max() * 1.3)
plt.sca(ax1)
plt.xticks(rotation=20, ha='right')

plt.tight_layout()
plt.show()

print('Exact counts:')
print(multi_counts.to_string())

### Insight: Class Distribution

**What does this chart tell us?**

- The **binary plot (left)** shows how many connections are normal (Benign) vs malicious (Attack).
  If the green bar is much taller than the red, the dataset is **imbalanced**. This is very common in
  cyber-security data because most real traffic is benign.
  An imbalanced dataset can make a model lazy: it learns to always predict "Benign" and still scores
  high accuracy, while missing real attacks.
  **Action:** We will use SMOTE oversampling in `train.py` to balance classes before training.

- The **multi-class plot (right)** breaks down every attack category individually.
  Attack types with very few samples (short bars) are hardest to detect.

**Key takeaway:** A Benign ratio above ~70% signals a class-imbalance problem that must be fixed.

---
## 3. Top 15 Features by Correlation with `is_attack`

In [ ]:
"""
Cell 4: Rank every numeric feature by absolute Pearson correlation
with the binary label `is_attack`, then draw a horizontal bar chart.
Positive = feature is higher in attack traffic.
Negative = feature is higher in benign traffic.
"""

corr_with_label = (
    df[NUM_COLS + ['is_attack']]
    .corr()['is_attack']
    .drop('is_attack')
    .drop('attack_type', errors='ignore')
    .sort_values(key=abs, ascending=False)
)

top_n    = min(15, len(corr_with_label))
top_corr = corr_with_label.head(top_n)
TOP_FEATURES = top_corr.index.tolist()  # used by later cells

bar_colors = ['#E74C3C' if v > 0 else '#3498DB' for v in top_corr.values]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    top_corr.index[::-1],
    top_corr.values[::-1],
    color=bar_colors[::-1],
    edgecolor='white', linewidth=0.6, height=0.65,
)

for bar, val in zip(bars, top_corr.values[::-1]):
    x_pos = val + (0.005 if val >= 0 else -0.005)
    ha    = 'left' if val >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            f'{val:+.3f}', va='center', ha=ha, fontsize=9, color='white')

ax.axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.6)
ax.set_xlabel('Pearson Correlation with is_attack')
ax.set_title(f'Top {top_n} Features by Absolute Correlation with is_attack',
             fontsize=13, fontweight='bold', pad=12)

legend_handles = [
    Patch(facecolor='#E74C3C', label='Higher in attacks'),
    Patch(facecolor='#3498DB', label='Higher in benign traffic'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9,
          framealpha=0.3, labelcolor='white')

plt.tight_layout()
plt.show()

print('Top features for next cells:', TOP_FEATURES)

### Insight: Feature Correlation Ranking

**What does this chart tell us?**

- Each bar shows how strongly one feature is *linearly related* to whether a connection is an attack.
  A bar pointing **right (red)** = that feature tends to be **larger** during attacks.
  A bar pointing **left (blue)** = that feature tends to be **smaller** during attacks.

- The **longer the bar, the more useful the feature** is for distinguishing attacks from normal traffic.
  Features with a very short bar barely differ between classes and might be safe to drop.

- This is a *linear* measure only. A feature with a low Pearson score can still be powerful
  if its relationship with the label is non-linear (e.g., "attack only if packets/s > 50,000").
  Tree-based models like XGBoost can capture those non-linear patterns automatically.

**Key takeaway:** The top features from this chart are our first candidate set for the model.
We will revisit importance after training using SHAP values for a more accurate picture.

---
## 4. Feature Distributions: Normal vs Attack

In [ ]:
"""
Cell 5: Overlapping KDE + histogram for each of the top features.
Green = Benign traffic, Red = Attack traffic.
Where distributions barely overlap, the feature is a strong discriminator.
"""

PLOT_FEATURES = TOP_FEATURES[:min(8, len(TOP_FEATURES))]

n_cols = 2
n_rows = int(np.ceil(len(PLOT_FEATURES) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3.5))
axes = np.array(axes).flatten()

benign_df = df[df['is_attack'] == 0]
attack_df = df[df['is_attack'] == 1]

for i, feat in enumerate(PLOT_FEATURES):
    ax = axes[i]

    # Clip extreme outliers at 1st and 99th percentile for readability
    clip_max = df[feat].quantile(0.99)
    clip_min = df[feat].quantile(0.01)

    b_data = benign_df[feat].clip(clip_min, clip_max)
    a_data = attack_df[feat].clip(clip_min, clip_max)

    # Histogram
    ax.hist(b_data, bins=40, density=True, alpha=0.35,
            color='#2ECC71', label='Benign')
    ax.hist(a_data, bins=40, density=True, alpha=0.35,
            color='#E74C3C', label='Attack')

    # KDE overlay
    try:
        b_data.plot.kde(ax=ax, color='#2ECC71', linewidth=2, label='')
        a_data.plot.kde(ax=ax, color='#E74C3C', linewidth=2, label='')
    except Exception:
        pass

    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8, framealpha=0.3, labelcolor='white')

# Hide unused subplots
for j in range(len(PLOT_FEATURES), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    'Feature Distributions: Benign vs Attack Traffic\n'
    '(values clipped to 1st-99th percentile to suppress outliers)',
    fontsize=13, fontweight='bold', y=1.01,
)
plt.tight_layout()
plt.show()

### Insight: Feature Distributions

**What does this chart tell us?**

Each plot shows the *shape* of a feature's values for benign (green) and attack (red) traffic separately.

- **Widely separated peaks** (green and red curves far apart): the feature is *very useful*.
  The model can draw a threshold between the two peaks to separate attacks from normal traffic.

- **Heavy overlap** (green and red sit on top of each other): the feature alone cannot distinguish
  attack from benign. It may still help *in combination* with other features, but it is weak alone.

- **Long right tails** are common in network data. DoS floods often have extremely high packet rates,
  so the tail of the red curve extends much further right than the green curve.
  Values are clipped at the 99th percentile here just to keep the charts readable.

**Key takeaway:** Features where the two colours are clearly separated are your strongest predictors.
If both distributions look the same for a feature, it is probably noise and could be dropped.

---
## 5. Correlation Heatmap of Top Features

In [ ]:
"""
Cell 6: Pearson correlation heatmap across the top features + is_attack.
Dark red  (+1) = strong positive correlation.
Dark blue (-1) = strong negative correlation.
White      (0) = no linear relationship.
"""

heatmap_cols = [c for c in TOP_FEATURES + ['is_attack']
                if c in df.select_dtypes(include='number').columns]

corr_matrix = df[heatmap_cols].corr()

# Mask upper triangle (mirror of lower triangle)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig_size = max(8, len(heatmap_cols))
fig, ax = plt.subplots(figsize=(fig_size, fig_size - 1))

cmap = sns.diverging_palette(220, 10, as_cmap=True)

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap=cmap,
    vmin=-1, vmax=1, center=0,
    annot=True, fmt='.2f', annot_kws={'size': 9},
    linewidths=0.5, linecolor='#0F1117',
    square=True,
    ax=ax,
)

ax.set_title('Pearson Correlation Heatmap: Top Features + is_attack',
             fontsize=13, fontweight='bold', pad=14)
ax.tick_params(axis='x', rotation=35, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)

plt.tight_layout()
plt.show()

### Insight: Correlation Heatmap

**What does this chart tell us?**

The heatmap is a grid where every square shows how strongly two features are correlated.
The **last row/column** (`is_attack`) shows each feature's relationship with the attack label.

| Colour | Value | Meaning |
|:------:|:-----:|:-------|
| Deep red | +1 | Both values go up together |
| White | 0 | No linear relationship |
| Deep blue | -1 | One goes up as the other goes down |

**Watch out for multicollinearity:**
If two *feature* columns (not the label) have a correlation close to +1 or -1, they carry nearly
the same information. Keeping both wastes resources and can confuse some models.
In `features.py` we will consider dropping one from each highly-correlated pair.

**Key takeaway:** Features with a deep red or blue square in the `is_attack` column are your strongest predictors.
Feature pairs with deep red/blue squares *between each other* are candidates for removal to reduce redundancy.

---
## EDA Summary

| Finding | Action |
|:--------|:-------|
| Dataset may be class-imbalanced | Apply SMOTE in `train.py` |
| Several features have high correlation with attacks | Prioritise these in `features.py` |
| Some feature pairs are highly correlated with each other | Drop one of each pair to reduce redundancy |
| Distributions differ visibly for top features | Model should find a clear decision boundary |

**Next script to implement:** `src/features.py`